In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [4]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [12]:
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel

load_dotenv()
openai_client = OpenAI()

class Questions(BaseModel):
    questions: list[str]

In [10]:
import json

documents_json = [json.dumps(doc) for doc in documents]
documents_json[:2]

['{"content": "# Introduction\\n\\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\\n\\nIn this module, we\'ll build a working Retrieval-Augmented\\nGeneration (RAG) system from scratch, step by step.\\n\\nWe write everything in plain Python. We build a small search index by\\nhand and call the LLM ourselves. I want you to see every piece first.\\nThat way you know what a framework does for you before you reach for\\none.\\n\\nPlaces where you can find me:\\n\\n- [My substack](https://alexeyondata.substack.com/)\\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\\n- [X](https://x.com/Al_Grigor)\\n\\n## LLMs\\n\\nAn LLM (Large Language Model) is a neural network trained on massive\\namounts of text. Given a prompt, it generates a continuation - a\\nplausible next piece of text.\\n\\nThink of your phone. When you type \\"how are\\" in WhatsApp, it suggests\\n\\"you\\" as the next word. \\"How are you\\" is the most commo

In [ ]:
## Q1
from evaluation_utils import llm_structured

total_usage = 0

for user_prompt in documents_json[:3]:
    result, usage = llm_structured(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )
    print("Input tokens: ", usage.input_tokens)
    total_usage += usage.input_tokens

total_usage / 3

Input tokens:  1020
Input tokens:  1286
Input tokens:  1753


1353.0

In [31]:
import pandas as pd

df_ground_truth = pd.read_csv("ground-truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")
df_ground_truth.head()

,question,filename
0,What exactly is a retrieval-augmented generati...,01-agentic-rag/lessons/01-intro.md
1,Why does this course build the RAG project in ...,01-agentic-rag/lessons/01-intro.md
2,What are the main weaknesses of large language...,01-agentic-rag/lessons/01-intro.md
3,What will the course build in the first part o...,01-agentic-rag/lessons/01-intro.md
4,What kind of example app are you building here...,01-agentic-rag/lessons/01-intro.md


In [28]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

295

In [61]:
from minsearch import Index, VectorSearch

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
index.fit(chunks)

def text_search(query, num_results=5):
    return index.search(
        query,
        num_results=num_results
    )

In [52]:
from embedder import Embedder

# Embed chunks
chunks_text = [chunk["content"] for chunk in chunks]
embedder = Embedder()
X = embedder.encode_batch(chunks_text)

# Build vector index
vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

def vector_search(query, num_results=5):
    query_vector = embedder.encode(query)

    return vindex.search(
        query_vector,
        num_results=num_results
    )

In [50]:
## Q2
q = ground_truth[0]["question"]
res = text_search(q)
res[0]

{'start': 3000,
 'content': 'we drop it.\n\nBuild a prompt that includes both the question and the context:\n\n```python\nprompt = f"""\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n```\n\nInstead of sending the raw question to the LLM, we send this prompt:\n\n```python\nanswer = llm(prompt)\nprint(answer)\n```\n\nAfter that, the answer is correct: "Yes, you can still join. If you want to\nreceive a certificate, you need to submit your project while\nsubmissions are still open."\n\nThis is the answer we actually want to give to our students. What we\njust did is nothing but RAG.\n\n## Retrieval plus generation\n\nRAG stands for Retrieval-Augmented Generation. Generation is the LLM\nproducing text, and retrieval is search. We retrieve 

In [53]:
## Q3
q = ground_truth[0]["question"]
res = vector_search(q)
res[0]

{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

In [54]:
from tqdm.auto import tqdm

def compute_relevance(q, search_function):
    doc_id = q["filename"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))

    return relevance

def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [55]:
## Q4
evaluate(
    ground_truth,
    text_search
)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}

In [56]:
## Q5
evaluate(
    ground_truth,
    vector_search
)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.725, 'mrr': 0.5486111111111112}

### Hybrid Search

In [71]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

def hybrid_search(query, k=60):
    # print(k)
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [ ]:
## Q6
for k_param in [1, 50, 100, 200]:
    func = lambda query: hybrid_search(query, k=k_param)

    res = {"k": k_param}
    eval_res = evaluate(
        ground_truth,
        func
    )
    res.update(eval_res)
    print(res)

  0%|          | 0/360 [00:00<?, ?it/s]

{'k': 1, 'hit_rate': 0.8388888888888889, 'mrr': 0.6481944444444449}


  0%|          | 0/360 [00:00<?, ?it/s]

{'k': 50, 'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}


  0%|          | 0/360 [00:00<?, ?it/s]

{'k': 100, 'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}


  0%|          | 0/360 [00:00<?, ?it/s]

{'k': 200, 'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}
